# SpecRAG LangGraph workflows

Updated: 2026-10-18

This notebook verifies the two workflow topologies defined in `specrag.orchestration`, then displays the supported business flow for Single Researcher and the runtime topology for Parallel Researcher. It uses compile-only test handlers: no graph is invoked, no corpus is loaded, and no model or provider is called.

Run **Run > Run All Cells** from the repository root with the project environment. The notebook displays both diagrams and exports standalone files under `artifacts/diagrams/`. PNG rendering uses Mermaid.ink; Mermaid source files are always exported as an offline fallback.

In [ ]:
from collections.abc import Mapping

from IPython.display import FileLink, Image, Markdown, display
from langchain_core.runnables.graph_mermaid import draw_mermaid_png

from specrag.config import REPO_ROOT
from specrag.models import ResearchTask, ResearchWorkerResult
from specrag.orchestration.parallel import build_parallel_research_workflow
from specrag.orchestration.state import ResearchState
from specrag.orchestration.workflow import (
    ResearchNodeHandlers,
    build_single_research_workflow,
)

## Compile-only handlers

Every handler raises if executed. Successful topology inspection therefore proves that graph compilation and rendering do not call application services or models.

In [ ]:
def test_node_handler(_state: ResearchState) -> Mapping[str, object]:
    raise AssertionError("topology inspection must not execute node handlers")


def test_parallel_researcher(
    _state: ResearchState,
    _task: ResearchTask,
    *,
    max_tool_calls: int,
) -> ResearchWorkerResult:
    del max_tool_calls
    raise AssertionError("topology inspection must not execute researcher workers")


handlers = ResearchNodeHandlers(
    router=test_node_handler,
    planner=test_node_handler,
    researcher=test_node_handler,
    source_validator=test_node_handler,
    sufficiency_judge=test_node_handler,
    replanner=test_node_handler,
    synthesizer=test_node_handler,
)

single_graph = build_single_research_workflow(handlers).graph
parallel_graph = build_parallel_research_workflow(
    handlers,
    test_parallel_researcher,
).graph

## Static topology contract

These assertions deliberately check nodes, edges, and whether an edge is conditional. A workflow change must update both this notebook and the automated topology tests.

In [ ]:
def topology(compiled_graph):
    drawable = compiled_graph.get_graph()
    nodes = set(drawable.nodes)
    edges = {
        (edge.source, edge.target, edge.conditional)
        for edge in drawable.edges
    }
    return nodes, edges


single_expected_nodes = {
    "__start__", "router", "planner", "researcher",
    "source_validator", "sufficiency_judge", "budget_guard",
    "replanner", "synthesizer", "__end__",
}
single_expected_edges = {
    ("__start__", "router", False),
    ("router", "planner", False),
    ("planner", "researcher", False),
    ("researcher", "source_validator", False),
    ("source_validator", "sufficiency_judge", False),
    ("sufficiency_judge", "budget_guard", False),
    ("budget_guard", "researcher", True),
    ("budget_guard", "replanner", True),
    ("budget_guard", "synthesizer", True),
    ("budget_guard", "__end__", True),
    ("replanner", "researcher", False),
    ("synthesizer", "__end__", False),
}

parallel_expected_nodes = {
    "__start__", "router", "planner", "dispatch",
    "researcher_worker", "fan_in", "source_validator",
    "sufficiency_judge", "budget_guard", "replanner",
    "synthesizer", "__end__",
}
parallel_expected_edges = {
    ("__start__", "router", False),
    ("router", "planner", False),
    ("planner", "dispatch", False),
    ("dispatch", "researcher_worker", True),
    ("dispatch", "budget_guard", True),
    ("researcher_worker", "fan_in", False),
    ("fan_in", "source_validator", False),
    ("source_validator", "sufficiency_judge", False),
    ("sufficiency_judge", "budget_guard", False),
    ("budget_guard", "dispatch", True),
    ("budget_guard", "replanner", True),
    ("budget_guard", "synthesizer", True),
    ("budget_guard", "__end__", True),
    ("replanner", "dispatch", False),
    ("synthesizer", "__end__", False),
}

assert topology(single_graph) == (single_expected_nodes, single_expected_edges)
assert topology(parallel_graph) == (parallel_expected_nodes, parallel_expected_edges)
print("Static topology verified; no handlers were executed.")

In [ ]:
OUTPUT_DIR = REPO_ROOT / "artifacts" / "diagrams"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def display_mermaid(mermaid: str, filename: str) -> None:
    mermaid_path = OUTPUT_DIR / f"{filename}.mmd"
    png_path = OUTPUT_DIR / f"{filename}.png"
    mermaid_path.write_text(mermaid, encoding="utf-8")
    try:
        png = draw_mermaid_png(
            mermaid,
            output_file_path=str(png_path),
            max_retries=3,
            retry_delay=2.0,
        )
        display(Image(png))
        display(FileLink(str(png_path), result_html_prefix="Saved PNG: "))
    except Exception as exc:
        display(Markdown(f"```mermaid\n{mermaid}\n```"))
        print(f"PNG rendering unavailable: {exc}")
    display(FileLink(str(mermaid_path), result_html_prefix="Saved Mermaid source: "))


def display_graph(compiled_graph, filename: str) -> None:
    display_mermaid(compiled_graph.get_graph().draw_mermaid(), filename)

## Single Researcher business workflow

This node-level view keeps the same granularity as the runtime stages. Router always enters Planner; internal budget checks and gap-task reactivation are represented by the single Judge-to-Researcher continuation loop rather than additional boxes.

In [ ]:
single_business_flow = (
    REPO_ROOT / "docs" / "architecture" / "single_research_workflow.mmd"
).read_text(encoding="utf-8")
display_mermaid(single_business_flow, "single-research-workflow")

Summary: the workflow moves from routing and planning through research, validation, and sufficiency checking; it either continues research, synthesizes an answer, or stops.

## Parallel Researcher workflow

In [ ]:
display_graph(parallel_graph, "parallel-research-workflow")

The parallel workflow dynamically sends ready tasks to researcher workers, merges their results at `fan_in`, and then applies the same validation, sufficiency, replan, and synthesis decisions.